## build_ewha_corpus

In [1]:
# 1. 패키지 설치
!pip install -qU langchain-upstage langchain-core langchain-community python-dotenv tqdm faiss-cpu

In [3]:
from google.colab import drive
import os

# 1. 드라이브 마운트
drive.mount('/content/drive')

# 2. 프로젝트 경로 설정
BASE_PATH = "/content/drive/MyDrive/rag-mmlu-ewha"
DATA_DIR = os.path.join(BASE_PATH, "data")
PDF_PATH = os.path.join(DATA_DIR, "ewha.pdf")
OUTPUT_PATH = os.path.join(DATA_DIR, "ewha_corpus.jsonl")

# 3. API 키 설정
from google.colab import userdata
os.environ["UPSTAGE_API_KEY"] = userdata.get('UPSTAGE_API_KEY')
print("Colab Secrets에서 API 키 로드 완료")

print(f"PDF 경로 확인: {PDF_PATH}")
if os.path.exists(PDF_PATH):
    print("✅ PDF 파일")
else:
    print("❌ PDF 파일 ")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab Secrets에서 API 키 로드 완료
PDF 경로 확인: /content/drive/MyDrive/rag-mmlu-ewha/data/ewha.pdf
✅ PDF 파일


In [4]:
import json
from tqdm.notebook import tqdm
from langchain_upstage import UpstageDocumentParseLoader, ChatUpstage
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def main():
    print("1. Upstage Document Parse로 PDF 구조 분석 시작")

    # [수정] output_type -> output_format 으로 변경
    # [참고] OCR은 Document Parse가 알아서 처리하므로 use_ocr 인자가 없으면 제거해도 됩니다.
    # 만약 output_format="html"에서도 에러가 나면 아예 줄을 지우세요 (기본값이 html/markdown임)

    loader = UpstageDocumentParseLoader(
        file_path=PDF_PATH,
        output_format="html",  # 여기가 핵심 수정 포인트! (type -> format)
        split="page",          # 페이지 단위 분할
        coordinates=False      # 좌표 정보는 필요 없으므로 끔 (속도 향상)
    )

    # API 호출
    try:
        raw_docs = loader.load()
        print(f"✅ PDF 파싱 완료! 총 {len(raw_docs)} 페이지")
    except TypeError as e:
        # 혹시 라이브러리 버전에 따라 output_format도 안 먹히면 제거하고 재시도
        print(f"옵션 에러 발생 ({e}), 기본 설정으로 재시도.")
        loader = UpstageDocumentParseLoader(file_path=PDF_PATH, split="page")
        raw_docs = loader.load()
        print(f"(기본 옵션) PDF 파싱 완료! 총 {len(raw_docs)} 페이지")

    # -------------------------------------------------------
    # 2. LLM Cleanup (문맥 연결 및 정제)
    # -------------------------------------------------------
    print("2. LLM Cleanup 시작 (줄바꿈 및 문맥 정리)")

    # Solar-Pro 사용 (강력 추천)
    llm = ChatUpstage(model="solar-pro", temperature=0)

    # ... (이하 코드는 동일)
    prompt = ChatPromptTemplate.from_template("""
    다음은 문서 파서(Document Parse)를 통해 추출된 텍스트(HTML 포함)입니다.
    불필요한 HTML 태그를 제거하고, 문맥상 끊어진 줄바꿈을 이어 자연스러운 텍스트로 변환하세요.

    [지침]
    1. <table> 태그 등 표 내용이 있다면, 텍스트로 풀어서 이해하기 쉽게 서술하세요.
    2. 조항 번호(제1조, 제2조...)와 날짜는 원본 형식을 유지하세요.
    3. 문장 중간에 부자연스럽게 끊긴 부분은 한 문장으로 이으세요.
    4. 오직 정제된 '텍스트' 내용만 출력하세요.

    [입력 데이터]:
    {text}

    [정제된 텍스트]:
    """)

    chain = prompt | llm | StrOutputParser()

    final_docs = []

    for i, doc in enumerate(tqdm(raw_docs, desc="Processing Pages")):
        if len(doc.page_content) < 20:
            continue

        try:
            cleaned_text = chain.invoke({"text": doc.page_content})

            meta = doc.metadata.copy()
            meta["doc_id"] = i + 1
            meta["section"] = f"Page {meta.get('page', i+1)}"

            final_docs.append({
                "doc_id": meta["doc_id"],
                "section": meta["section"],
                "text": cleaned_text
            })

        except Exception as e:
            print(f"⚠️ Error on page {i}: {e}")
            final_docs.append({
                "doc_id": i + 1,
                "section": f"Page {i+1}",
                "text": doc.page_content
            })

    # -------------------------------------------------------
    # 3. JSONL 저장
    # -------------------------------------------------------
    print(f"3. 결과 저장 중 ({OUTPUT_PATH})")
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for row in final_docs:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("모든 작업 완료! ewha_corpus.jsonl 파일이 생성되었습니다.")

if __name__ == "__main__":
    main()

1. Upstage Document Parse로 PDF 구조 분석 시작
✅ PDF 파싱 완료! 총 54 페이지
2. LLM Cleanup 시작 (줄바꿈 및 문맥 정리)


Processing Pages:   0%|          | 0/54 [00:00<?, ?it/s]

3. 결과 저장 중 (/content/drive/MyDrive/rag-mmlu-ewha/data/ewha_corpus.jsonl)
모든 작업 완료! ewha_corpus.jsonl 파일이 생성되었습니다.


In [5]:
import json
import pandas as pd

print("📂 생성된 Corpus 파일 내용 미리보기\n" + "="*50)
# 1. 보기 편하게 Pandas 데이터프레임으로 상위 5개 출력
try:
    df = pd.read_json(OUTPUT_PATH, lines=True)
    print(f"총 데이터 개수: {len(df)}개 (페이지 단위)")
    display(df.head()) # 표 형태로 깔끔하게 보기
except Exception as e:
    print(f"Pandas 로딩 실패: {e}")

print("\n\n [상세 텍스트 확인] 첫 번째 & 중간 데이터 샘플")
print("="*50)

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

    # 첫 번째 페이지 (보통 서문이나 목차)
    first_doc = json.loads(lines[0])
    print(f"\n [ID: {first_doc['doc_id']}] Section: {first_doc['section']}")
    print("-" * 20)
    print(first_doc['text'][:500]) # 앞 500자만 출력
    print("..." + "\n")

    # 중간 페이지 (보통 본문이나 표) - 10번째 페이지 확인
    if len(lines) > 10:
        mid_doc = json.loads(lines[10])
        print(f" [ID: {mid_doc['doc_id']}] Section: {mid_doc['section']}")
        print("-" * 20)
        print(mid_doc['text'][:500])
        print("...")

📂 생성된 Corpus 파일 내용 미리보기
총 데이터 개수: 54개 (페이지 단위)


,doc_id,section,text
0,1,Page 1,이화여자대학교 학칙 \n1946. 8. 15. 제정 \n2017. 8. 16. ...
1,2,Page 2,이화여자대학교 학칙 \n⑤ 전공별 선발로 입학한 학생은 전공별 신청자격 및 승인요...
2,3,Page 3,이화여자대학교 학칙 \n학기로 나눈다. \n② 제1항의 일반학기 외에 하기방학과...
3,4,Page 4,이화여자대학교 학칙(개정 1998.5.22) \n③ 기초·소양교육을 2년 이상 수...
4,5,Page 5,이화여자대학교 학칙 \n제19조(입학금 등) ① 입학이 허가된 자는 정해진 기일 ...




 [상세 텍스트 확인] 첫 번째 & 중간 데이터 샘플

 [ID: 1] Section: Page 1
--------------------
이화여자대학교 학칙  
1946. 8. 15. 제정  
2017. 8. 16. 개정  

제1장 총칙  
제1조(목적) 본교는 대한민국의 교육이념과 기독교정신을 바탕으로 하여 학술의 깊은 이론과 그 광범하고 정밀한 응용방법을 교수·연구하며, 인격을 도야하여 국가와 인류사회의 발전에 공헌할 수 있는 지도여성을 양성함을 목적으로 한다.  
제2조(명칭) 본교는 이화여자대학교라 부른다.  
제3조(위치) 본교는 서울특별시 서대문구 이화여대길 52에 둔다. (개정 2013.2.25.)  

제2장 편제  
제4조(대학 및 대학원) ① 본교에는 다음 각 호의 대학을 둔다.  
1. 인문과학대학, 사회과학대학, 자연과학대학, 엘텍공과대학, 음악대학, 조형예술대학, 사범대학, 경영대학, 신산업융합대학, 의과대학, 간호대학, 약학대학, 스크랜튼대학(이하 “각 대학”이라 한다) (개정 2016.6.16.)  
2. 호크마(HOKMA)교양대학  
② 본교에는 대학원, 국제대학원, 통역번역대학원, 경영
...

 [ID: 11] Section: Page 11
--------------------
이화여자대학교 학칙  
1. 교과과정 운영상 또는 기타 특별한 사유가 있어 총장의 승인을 얻은 경우: 21학점까지 취득  
2. 직전 학기(계절학기를 제외한다)말의 평균성적이 3.75 이상인 경우: 21학점까지 취득  
3. 제15조제2항에 따라 편입학(이하 ‘학사편입학’이라 한다)한 경우: 21학점까지 취득  
4. 학·석사 연계과정을 이수하는 경우: 21학점까지 취득  
5. 의과대학 의학과의 경우: 24학점까지 취득  
6. 약학대학의 경우: 21학점까지 취득(다만, ‘기독교와 세계’ 교과목을 수강하는 학기 및 인문학 관련 교양과목을 수강하는 경우, 최대 3학기에 한하여 24학점까지 취득)  
② 제1항의 규정에도 불구하고 학기당 취득기준학점인

In [6]:
import json
import re
import os

# 1. 파일 경로 설정
BASE_PATH = "/content/drive/MyDrive/rag-mmlu-ewha/data"
INPUT_PATH = os.path.join(BASE_PATH, "ewha_corpus.jsonl")
OUTPUT_PATH = os.path.join(BASE_PATH, "ewha_corpus_structured.jsonl")

def restructure_corpus():
    print(f"[Info] 데이터 재구조화 시작: {INPUT_PATH} -> {OUTPUT_PATH}")

    # 1. 기존 페이지 단위 텍스트 모두 합치기
    if not os.path.exists(INPUT_PATH):
        print(f"[Error] 입력 파일을 찾을 수 없습니다: {INPUT_PATH}")
        return

    full_text = ""
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        # doc_id 순서대로 정렬하여 읽기
        docs = [json.loads(line) for line in f]
        docs.sort(key=lambda x: x['doc_id'])

        for doc in docs:
            # 페이지 간 경계에 줄바꿈 추가
            full_text += doc['text'] + "\n"

    print(f"[Info] 전체 텍스트 병합 완료 (총 길이: {len(full_text)}자)")

    # 2. 정규식으로 '제N장' 또는 '[별표 N]' 패턴 기준으로 분리
    # 패턴: 줄바꿈+제+숫자+장 또는 줄바꿈+[별표+숫자]
    split_pattern = r"(\n\s*제\s*\d+\s*장|\n\s*\[별표\s*\d+\])"

    parts = re.split(split_pattern, full_text)

    new_docs = []

    # 첫 번째 부분 (서문 등 헤더 이전 내용)
    if parts[0].strip():
        new_docs.append({
            "doc_id": 1,
            "section": "서문/총칙 이전",
            "text": parts[0].strip()
        })

    current_doc_id = 2

    # 3. 헤더(구분자)와 내용 합쳐서 저장
    # parts 리스트: [서문, "제1장", "내용...", "제2장", "내용...", ...]
    count_chapters = 0
    for i in range(1, len(parts), 2):
        if i + 1 < len(parts):
            header = parts[i].strip()
            content = parts[i+1].strip()

            # 섹션명 생성 (헤더 + 첫 줄 내용 일부)
            first_line = content.split('\n')[0]
            if len(first_line) > 50:
                first_line = first_line[:50] + "..."
            section_title = f"{header} {first_line}"

            combined_text = f"{header}\n{content}"

            new_docs.append({
                "doc_id": current_doc_id,
                "section": section_title,
                "text": combined_text
            })
            current_doc_id += 1
            count_chapters += 1

    print(f"[Info] 재구조화 완료. 총 {count_chapters}개의 장(Section)으로 분할됨.")

    # 4. 저장
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for doc in new_docs:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")

    print(f"[Success] 저장 완료: {OUTPUT_PATH}")

if __name__ == "__main__":
    restructure_corpus()

[Info] 데이터 재구조화 시작: /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_corpus.jsonl -> /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_corpus_structured.jsonl
[Info] 전체 텍스트 병합 완료 (총 길이: 52310자)
[Info] 재구조화 완료. 총 25개의 장(Section)으로 분할됨.
[Success] 저장 완료: /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_corpus_structured.jsonl


In [7]:
import json
import os

# 파일 경로
BASE_PATH = "/content/drive/MyDrive/rag-mmlu-ewha/data"
STRUCTURED_PATH = os.path.join(BASE_PATH, "ewha_corpus_structured.jsonl")

def inspect_samples():
    print(f"[Info] 파일 읽기: {STRUCTURED_PATH}")

    if not os.path.exists(STRUCTURED_PATH):
        print("[Error] 파일을 찾을 수 없습니다.")
        return

    data = []
    with open(STRUCTURED_PATH, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))

    total_docs = len(data)
    print(f"[Info] 총 문서 개수: {total_docs}개 (장/별표 단위)")

    # 확인할 인덱스 지정 (처음, 중간, 끝)
    indices_to_check = [0, 2, 5, total_docs - 2]

    for idx in indices_to_check:
        if idx >= total_docs: continue

        doc = data[idx]
        print("\n" + "="*60)
        print(f"Doc ID: {doc['doc_id']} | Index: {idx}")
        print(f"Section: {doc['section']}")
        print("-" * 60)
        # 내용이 너무 길면 앞 300자, 뒤 100자만 출력
        text = doc['text']
        if len(text) > 400:
            print(text[:300] + "\n\n... (중략) ...\n\n" + text[-100:])
        else:
            print(text)

if __name__ == "__main__":
    inspect_samples()

[Info] 파일 읽기: /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_corpus_structured.jsonl
[Info] 총 문서 개수: 26개 (장/별표 단위)

Doc ID: 1 | Index: 0
Section: 서문/총칙 이전
------------------------------------------------------------
이화여자대학교 학칙  
1946. 8. 15. 제정  
2017. 8. 16. 개정

Doc ID: 3 | Index: 2
Section: 제2장 편제  
------------------------------------------------------------
제2장
편제  
제4조(대학 및 대학원) ① 본교에는 다음 각 호의 대학을 둔다.  
1. 인문과학대학, 사회과학대학, 자연과학대학, 엘텍공과대학, 음악대학, 조형예술대학, 사범대학, 경영대학, 신산업융합대학, 의과대학, 간호대학, 약학대학, 스크랜튼대학(이하 “각 대학”이라 한다) (개정 2016.6.16.)  
2. 호크마(HOKMA)교양대학  
② 본교에는 대학원, 국제대학원, 통역번역대학원, 경영전문대학원, 법학전문대학원, 교육대학원, 디자인대학원, 사회복지대학원, 신학대학원, 정책과학대학원, 공연예술대학원, 임상보건융합대학

... (중략) ...

래평생교육원의 학칙은 따로 정한다. (개정 2015.11.27.)  
[본조신설 1986.1.7.]  
[제목개정 2015.11.27.]  
제6조의3 삭제 (2016.2.26.)

Doc ID: 6 | Index: 5
Section: 제5장 입학, 편입학 및 등록  
------------------------------------------------------------
제5장
입학, 편입학 및 등록  
제13조(입학시기) 입학시기는 학년개시일로부터 30일 이내로 한다. 다만, 재입학, 편입학 및 외국인(외국국적을 보유한 자로서 부모가 모두 외국인인 자)의 신입

## build_ewha_retrieval

In [8]:
# =========================================================
# [수정본] 긴 문서 자동 분할(Chunking) 기능 추가
# =========================================================

import os
import json
import numpy as np
import faiss
import pandas as pd
from tqdm.notebook import tqdm
from langchain_upstage import UpstageEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter # 쪼개기 도구

# ---------------------------------------------------------
# [설정] 경로
# ---------------------------------------------------------
BASE_PATH = "/content/drive/MyDrive/rag-mmlu-ewha/data"
CORPUS_PATH = os.path.join(BASE_PATH, "ewha_corpus_structured.jsonl")
INDEX_PATH = os.path.join(BASE_PATH, "ewha_index.faiss")
TESTSET_PATH = os.path.join(BASE_PATH, "testset.csv")

# =========================================================
# 1. Upstage 임베딩 모델 래퍼
#    - 문서: embed_documents()
#    - 쿼리: embed_query()
#    - 벡터는 L2 정규화해서 코사인 유사도(IP)로 사용
# =========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

class UpstageEmbeddingModel:
    def __init__(self, model_name: str = "solar-embedding-1-large"):
        print(f"[Init] Upstage Embedding Model: {model_name}")
        self.model = UpstageEmbeddings(model=model_name)
        self.dim = None

        # 문단/문장 단위로 먼저 잘라주는 splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,      # 대략 지금이랑 비슷한 크기
            chunk_overlap=100,    # 문맥 겹치기
            separators=["\n\n", "\n", " ", ""],
        )

    def _chunk_text(self, text: str):
        text = text.strip()
        if not text:
            return []
        return self.text_splitter.split_text(text)

    def encode_documents(self, texts, batch_size: int = 32) -> np.ndarray:
        """
        코퍼스(각 섹션)를 임베딩.
        - 섹션 하나를 여러 chunk로 나눈 후 embed_documents로 임베딩
        - 섹션 벡터 = chunk 임베딩들의 평균
        """
        doc_vectors = []

        for idx, text in enumerate(tqdm(texts, desc="Encoding corpus (by section)")):
            #chunks = self._chunk_text(text, max_chars=1000)  # 안전하게 잘게
            chunks = self._chunk_text(text)
            if not chunks:
                continue
            # 한 섹션 안의 chunk들을 한 번에 임베딩
            emb_chunks = self.model.embed_documents(chunks)  # List[List[float]]
            emb_chunks = np.array(emb_chunks, dtype="float32")  # (num_chunks, dim)

            # 섹션 대표 벡터 = chunk 벡터들의 평균
            doc_vec = emb_chunks.mean(axis=0)
            doc_vectors.append(doc_vec)

        arr = np.vstack(doc_vectors)  # (num_sections, dim)
        print(f"[Embedding] Corpus vectors shape(before norm): {arr.shape}")

        # L2 정규화 → IP == cosine similarity
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        arr = arr / np.clip(norms, 1e-12, None)

        if self.dim is None:
            self.dim = arr.shape[1]
        elif self.dim != arr.shape[1]:
            raise ValueError(f"Document dim changed: {self.dim} vs {arr.shape[1]}")

        print("→ 첫 5개 섹션 벡터 L2 norm:", np.linalg.norm(arr[:5], axis=1))
        return arr

    def encode_query(self, text: str) -> np.ndarray:
        """
        쿼리(질문) 임베딩: embed_query 사용 + L2 정규화
        """
        vec = self.model.embed_query(text)
        arr = np.array(vec, dtype="float32").reshape(1, -1)

        # L2 정규화
        norm = np.linalg.norm(arr, axis=1, keepdims=True)
        arr = arr / np.clip(norm, 1e-12, None)

        if self.dim is not None and arr.shape[1] != self.dim:
            raise ValueError(
                f"Query dim ({arr.shape[1]}) != corpus dim ({self.dim}). "
                "모델 이름/설정이 일치하는지 확인하세요."
            )

        print("→ q_vec L2 norm:", np.linalg.norm(arr))
        return arr

# =========================================================
# 2. FAISS 벡터 스토어
# =========================================================

class FaissVectorStore:
    def __init__(self, dim: int, index_path: str):
        self.dim = dim
        self.index_path = index_path
        # L2 정규화된 벡터 + IP = cosine similarity
        self.index = faiss.IndexFlatIP(dim)

    def build_index(self, vectors: np.ndarray):
        print(f"[Build] Building index... shape={vectors.shape}")
        self.index.add(vectors)
        print(f"[Build] Total vectors in index: {self.index.ntotal}")

    def save(self):
        os.makedirs(os.path.dirname(self.index_path), exist_ok=True)
        faiss.write_index(self.index, self.index_path)
        print(f"[Save] Index saved → {self.index_path}")

    def load(self):
        if not os.path.exists(self.index_path):
            raise FileNotFoundError(f"Index file not found: {self.index_path}")
        self.index = faiss.read_index(self.index_path)
        print(f"[Load] Index loaded. Total vectors: {self.index.ntotal}")

    def search(self, q_emb: np.ndarray, top_k: int = 5):
        assert q_emb.ndim == 2, f"q_emb must be 2D, got shape {q_emb.shape}"
        assert q_emb.shape[1] == self.index.d, f"query dim {q_emb.shape[1]} != index dim {self.index.d}"
        D, I = self.index.search(q_emb, top_k)
        return D[0], I[0]

# =========================================================
# 3. Ewha structured corpus 로딩
#    (restructure_corpus 돌려서 만든 파일)
# =========================================================

BASE_PATH = "/content/drive/MyDrive/rag-mmlu-ewha/data"
CORPUS_PATH = os.path.join(BASE_PATH, "ewha_corpus_structured.jsonl")
INDEX_PATH = os.path.join(BASE_PATH, "ewha_index.faiss")
TESTSET_PATH = os.path.join(BASE_PATH, "testset.csv")

print("1. Structured corpus 로딩")

corpus_texts = []
corpus_sections = []

if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            corpus_texts.append(obj["text"])
            corpus_sections.append(obj.get("section", ""))
    print(f"✅ 로드 완료: {len(corpus_texts)}개 섹션")
else:
    raise FileNotFoundError(f"❌ {CORPUS_PATH} 를 찾을 수 없습니다. 먼저 restructure_corpus()를 실행하세요.")

# =========================================================
# 4. 코퍼스 임베딩 + FAISS 인덱스 생성
# =========================================================

print("\n 2. Upstage 임베딩 생성")
embedder = UpstageEmbeddingModel()

corpus_vectors = embedder.encode_documents(corpus_texts, batch_size=32)
print("→ corpus_vectors.shape:", corpus_vectors.shape)

print("\n 3. FAISS 인덱스 구축 및 저장")
store = FaissVectorStore(dim=corpus_vectors.shape[1], index_path=INDEX_PATH)
store.build_index(corpus_vectors)
store.save()

1. Structured corpus 로딩
✅ 로드 완료: 26개 섹션

 2. Upstage 임베딩 생성
[Init] Upstage Embedding Model: solar-embedding-1-large


Encoding corpus (by section):   0%|          | 0/26 [00:00<?, ?it/s]

[Embedding] Corpus vectors shape(before norm): (26, 4096)
→ 첫 5개 섹션 벡터 L2 norm: [1.         1.         0.99999994 1.         1.        ]
→ corpus_vectors.shape: (26, 4096)

 3. FAISS 인덱스 구축 및 저장
[Build] Building index... shape=(26, 4096)
[Build] Total vectors in index: 26
[Save] Index saved → /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_index.faiss


In [9]:
# =========================================================
# 5. testset.csv로 실제 검색 성능 확인
#    - 첫 문제의 prompts 전체를 쿼리로 사용
# =========================================================

print("\n 4. Testset 기반 검색 성능 확인")

if not os.path.exists(TESTSET_PATH):
    print(f"testset.csv ({TESTSET_PATH})를 찾을 수 없습니다.")
else:
    test_df = pd.read_csv(TESTSET_PATH)
    q1 = test_df.iloc[0]["prompts"]

    print("\n[QUERY]")
    print(q1)

    # 질문 임베딩 (embed_query 사용 + L2 정규화)
    q_vec = embedder.encode_query(q1)
    print("→ q_vec.shape:", q_vec.shape)

    # 인덱스 로드 (dim은 corpus_vectors.shape[1]과 동일)
    store = FaissVectorStore(dim=corpus_vectors.shape[1], index_path=INDEX_PATH)
    store.load()

    # 검색
    scores, idxs = store.search(q_vec, top_k=5)

    # raw score / 최대값 같이 보기
    print("\nraw scores:", scores)
    print("max |score|:", float(np.max(np.abs(scores))))

    print("\n=== TOP 5 검색 결과 ===")
    for rank, i in enumerate(idxs):
        print(f"[{rank+1}] score={scores[rank]:.6f}")  # 소수점 6자리
        print(f"Section: {corpus_sections[i]}")
        print(corpus_texts[i][:300], "...")
        print("-" * 80)



 4. Testset 기반 검색 성능 확인

[QUERY]
QUESTION1) 재학 중인 학생이 휴학을 하려면 학기 개시일로부터 며칠 이내에 휴학을 신청하야하나요?
(A) 30일
(B) 45일 
(C) 60일
(D) 90일
→ q_vec L2 norm: 1.0
→ q_vec.shape: (1, 4096)
[Load] Index loaded. Total vectors: 26

raw scores: [0.62721837 0.5253399  0.52014256 0.5158283  0.50934166]
max |score|: 0.6272183656692505

=== TOP 5 검색 결과 ===
[1] score=0.627218
Section: 제8장 휴학, 복학, 제적, 자퇴 및 재입학 (개정 2017.8.16.)  
제8장
휴학, 복학, 제적, 자퇴 및 재입학 (개정 2017.8.16.)  
제26조(휴학) ① 질병 기타 부득이한 사정으로 3주일 이상 수강할 수 없는 자는 총장의 허가를 얻어 휴학할 수 있다.  
② 총장은 건강상의 이유로 정상적인 수업을 받을 수 없다고 인정되는 자에 대하여 휴학을 명할 수 있다. (개정 1988.7.28)  
③ 1회의 휴학기간은 1년 이내로 한다. 다만, 교과과정상의 필요에 따라 총장이 지정하는 학부, 학과 또는 전공에 있어서는 이를 1년으로 한다. (개정 1996.2.15)  
④ 휴학기간은 통산하여 3년 ...
--------------------------------------------------------------------------------
[2] score=0.525340
Section: 제4장 학년, 학기, 수업일수, 휴업일 및 교원의 교수시간 (개정 1998.6.23)  
제4장
학년, 학기, 수업일수, 휴업일 및 교원의 교수시간 (개정 1998.6.23)  
제10조(학년, 학기) ① 학년은 3월 1일부터 다음해 2월 말일까지로 하고 이를 제1학기와 제2학기로 구분한다.  

2 - 2 - 2
이화여자

## Retrieval Check

In [12]:
import os
import re
import pandas as pd

from google.colab import files
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 파일 업로드
uploaded = files.upload()

# 업로드된 파일명 확인 및 로드
filename = list(uploaded.keys())[0]
print(f"[INFO] File loaded: {filename}")

Saving test_ewha_num1.csv to test_ewha_num1 (2).csv
[INFO] File loaded: test_ewha_num1 (2).csv


In [14]:
# ---------------------------------------------------------
# 2. Solar LLM 설정 (정답 생성용)
#     - embedder, store, corpus_texts, corpus_sections 는
#       앞에서 이미 만든 상태라고 가정
# ---------------------------------------------------------
print("Solar LLM 초기화 중...")
llm = ChatUpstage(model="solar-pro2")  # 필요시 "solar-pro2" 등으로 변경

rag_prompt = ChatPromptTemplate.from_template("""
당신은 이화여자대학교 학칙에 정통한 AI 조교입니다.
아래 [관련 학칙]을 바탕으로 사용자의 질문에 대한 정답 기호를 선택하세요.

[관련 학칙]
{context}

[질문]
{question}

[지시사항]
1. 반드시 제공된 [관련 학칙]에 근거하여 정답을 고르세요.
2. 정답 기호(예: (A), (B), (C), (D)...)만 명확하게 출력하세요.
3. 부연 설명은 하지 마세요.

정답:
""")

rag_chain = rag_prompt | llm | StrOutputParser()

# ---------------------------------------------------------
# 3. 평가 함수 정의
# ---------------------------------------------------------
def evaluate_accuracy(testset_path, embedder, store, top_k=3):
    print(f"\n평가 시작: {testset_path}")

    if not os.path.exists(testset_path):
        print(f"파일 없음: {testset_path}")
        return

    df = pd.read_csv(testset_path)
    total_questions = len(df)
    correct_count = 0
    results = []

    print(f"총 {total_questions}개의 문제에 대해 테스트를 진행합니다.\n")

    for idx, row in df.iterrows():
        question = row["prompts"]
        ground_truth = row["answers"].strip()  # 예: "(A)"

        # 1) 검색 (Retrieve)
        q_vec = embedder.encode_query(question)
        scores, idxs = store.search(q_vec, top_k=top_k)

        context_list = [corpus_texts[i] for i in idxs]
        context_text = "\n\n".join(context_list)

        # 2) 답변 생성 (Generate)
        try:
            ai_answer = rag_chain.invoke(
                {"context": context_text, "question": question}
            ).strip()

            # 3) 정답 기호만 추출
            m = re.search(r"\([A-J]\)", ai_answer)
            if m:
                predicted = m.group(0)
            else:
                # 예외적으로 괄호 없이 A,B...만 나온 경우 대응
                # 앞쪽 3글자 정도만 잘라서 보정
                candidate = ai_answer.strip().upper()[:3]
                # "(A", "A)", "A" 등일 수 있음
                # 모두 "(A)" 형태로 통일
                letter_m = re.search(r"[A-J]", candidate)
                if letter_m:
                    predicted = f"({letter_m.group(0)})"
                else:
                    predicted = "?"

        except Exception as e:
            print(f"문제 {idx+1} 처리 중 오류: {e}")
            predicted = "Error"

        # 4) 채점
        is_correct = (predicted == ground_truth)
        if is_correct:
            correct_count += 1
            result_mark = "정답"
        else:
            result_mark = "오답"

        results.append({
            "No": idx + 1,
            "Question": question.split('\n')[0][:40] + "...",
            "Real": ground_truth,
            "Pred": predicted,
            "Result": result_mark,
        })

        # 진행 상황 출력
        if (idx + 1) % 5 == 0:
            print(f"[{idx+1}/{total_questions}] 진행 중... 현재 정답 수: {correct_count}")

    # -----------------------------------------------------
    # 5. 최종 리포트
    # -----------------------------------------------------
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0.0

    print("\n" + "=" * 60)
    print("최종 결과 리포트")
    print("=" * 60)
    print(f"총 문제 수 : {total_questions}")
    print(f"정답 수    : {correct_count}")
    print(f"정확도(Acc): {accuracy:.2f}%")
    print("=" * 60)

    # 필요하면 DataFrame으로 결과 확인
    # result_df = pd.DataFrame(results)
    # display(result_df)

# ---------------------------------------------------------
# 4. 실행
# ---------------------------------------------------------
evaluate_accuracy(filename, embedder, store, top_k=3)

Solar LLM 초기화 중...

평가 시작: test_ewha_num1 (2).csv
총 35개의 문제에 대해 테스트를 진행합니다.

→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
[5/35] 진행 중... 현재 정답 수: 3
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
[10/35] 진행 중... 현재 정답 수: 7
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 0.99999994
[15/35] 진행 중... 현재 정답 수: 12
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 0.99999994
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
[20/35] 진행 중... 현재 정답 수: 17
→ q_vec L2 norm: 1.0000001
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0000001
→ q_vec L2 norm: 1.0
[25/35] 진행 중... 현재 정답 수: 20
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 1.0
→ q_vec L2 norm: 0.99999994
→ q_vec L2 norm: 0.99999994
→ q_vec L2 norm: 1.0
[30/35] 진행 중... 현재 정답 수: 21
→ q_vec L2 norm: 1.0000001
→ q_vec L2 norm: 1.0000001
→ q_vec L2 norm: 1.0
→ q_vec L2 no